# Instalação  e configuração

In [ ]:
%pip install ultralytics pyyaml --no-deps

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
%pip install torchvision

Defaulting to user installation because normal site-packages is not writeable
  Using cached torchvision-0.29.2-py3-none-win_amd64.whl.metadata (5.7 kB)
  Using cached torch-2.15.0-cp313-cp313-win_amd64.whl.metadata (36 kB)
Using cached torchvision-0.29.2-py3-none-win_amd64.whl (1.4 MB)
Using cached torch-2.15.0-cp313-cp313-win_amd64.whl (126.7 MB)

  Attempting uninstall: torch

    Found existing installation: torch 2.11.0

   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
    Uninstalling torch-2.11.0:
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   --------------------------------

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\User\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\torch-2.15.0.dist-info\\licenses\\third_party\\cudnn_frontend\\python\\cudnn\\moe_ep\\_megamoe_backend\\cutedsl_src\\LICENSE.Apache-2.0'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Validação de imagens do dataset

Nesta etapa, verificamos:
- imagens sem arquivo `.txt`;
- anotações sem imagem correspondente;
- imagens corrompidas;
- formato YOLO com cinco campos;
- classes 0 e 1;
- coordenadas normalizadas entre 0 e 1;
- quantidade de imagens positivas e negativas.

In [ ]:
%pip install pillow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:

from pathlib import Path
from collections import Counter
from PIL import Image, UnidentifiedImageError

# Localiza a raiz do projeto
current = Path.cwd()

PROJECT_ROOT = next(
    (p for p in [current, *current.parents] if p.name == "urbansense"),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Pasta raiz 'urbansense' não encontrada.")

DATASET = PROJECT_ROOT / "data" / "dataset"

print("Projeto:", PROJECT_ROOT)
print("Dataset:", DATASET.resolve())

if not DATASET.exists():
    raise FileNotFoundError(f"Dataset não encontrado: {DATASET}")

# Localiza imagens e anotações em todas as subpastas
images = sorted(
    p for p in DATASET.rglob("*")
    if p.suffix.lower() in (".jpg", ".jpeg")
)

labels = sorted(
    p for p in DATASET.rglob("*")
    if p.suffix.lower() == ".txt"
)

missing_labels = []
missing_images = []
invalid_labels = []
corrupt_images = []
class_counts = Counter()

images_with_objects = 0
images_without_objects = 0
invalid_coordinates = 0

# Verifica imagens e respectivas anotações
for image_path in images:
    label_path = image_path.with_suffix(".txt")

    if not label_path.exists():
        missing_labels.append(str(image_path))
        continue

    try:
        with Image.open(image_path) as image:
            image.verify()
    except (UnidentifiedImageError, OSError):
        corrupt_images.append(str(image_path))

    lines = [
        line.strip()
        for line in label_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    if not lines:
        images_without_objects += 1
    else:
        images_with_objects += 1

    for line_number, line in enumerate(lines, start=1):
        fields = line.split()

        if len(fields) != 5:
            invalid_labels.append(
                f"{label_path}:{line_number} - campos: {len(fields)}"
            )
            continue

        try:
            class_id = int(fields[0])
            x, y, width, height = map(float, fields[1:])
        except ValueError:
            invalid_labels.append(
                f"{label_path}:{line_number} - valor inválido"
            )
            continue

        if class_id not in (0, 1):
            invalid_labels.append(
                f"{label_path}:{line_number} - classe inválida: {class_id}"
            )

        if not all(
            0 <= value <= 1
            for value in (x, y, width, height)
        ):
            invalid_coordinates += 1
            invalid_labels.append(
                f"{label_path}:{line_number} - coordenada fora de [0, 1]"
            )

        if width <= 0 or height <= 0:
            invalid_labels.append(
                f"{label_path}:{line_number} - largura/altura inválida"
            )

        class_counts[class_id] += 1

# Verifica anotações sem imagem correspondente
image_stems = {
    (p.parent, p.stem)
    for p in images
}

missing_images = [
    str(path)
    for path in labels
    if (path.parent, path.stem) not in image_stems
]

# Resultados
print("\n========== RESULTADOS ==========")
print("Imagens:", len(images))
print("Anotações:", len(labels))
print("Imagens com objetos:", images_with_objects)
print("Imagens negativas:", images_without_objects)
print("Caixas por classe:", dict(class_counts))
print("Imagens sem anotação:", len(missing_labels))
print("Anotações sem imagem:", len(missing_images))
print("Imagens corrompidas:", len(corrupt_images))
print("Anotações inválidas:", len(invalid_labels))
print("Coordenadas fora do intervalo:", invalid_coordinates)

# Validação final
assert not missing_labels, "Existem imagens sem arquivo de anotação"
assert not missing_images, "Existem anotações sem imagem correspondente"
assert not corrupt_images, "Existem imagens corrompidas"
assert not invalid_labels, "Existem anotações inválidas"


Projeto: c:\Workspace\Study\UPE\urbansense
Dataset: C:\Workspace\Study\UPE\urbansense\data\dataset

========== RESULTADOS ==========
Imagens: 2099
Anotações: 2099
Imagens com objetos: 1510
Imagens negativas: 589
Caixas por classe: {0: 3166, 1: 442}
Imagens sem anotação: 0
Anotações sem imagem: 0
Imagens corrompidas: 0
Anotações inválidas: 0
Coordenadas fora do intervalo: 0


# Divisão treino/validação/teste

70/15/15.

In [ ]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import shutil

DATASET_SAIDA = DATASET.parent / "Potholes_dataset_dividido"
SEMENTE = 42

images = sorted(DATASET.rglob("*.jpg"))
print("Imagens para dividir:", len(images))

registros = []

for imagem in images:
    anotacao = imagem.with_suffix(".txt")
    classes = sorted({
        linha.split()[0]
        for linha in anotacao.read_text().splitlines()
        if linha.strip()
    })
    assinatura = "|".join(classes) if classes else "sem_objeto"
    registros.append((imagem, assinatura))

print("Registros para dividir:", len(registros))

assinaturas = [assinatura for _, assinatura in registros]

treino, temporario = train_test_split(
    registros,
    test_size=0.30,
    random_state=SEMENTE,
    stratify=assinaturas
)

assinaturas_temporarias = [assinatura for _, assinatura in temporario]

validacao, teste = train_test_split(
    temporario,
    test_size=0.50,
    random_state=SEMENTE,
    stratify=assinaturas_temporarias
)

particoes = {
    "train": treino,
    "val": validacao,
    "test": teste,
}

for nome, registros_particao in particoes.items():
    for imagem, _ in registros_particao:
        caminho_relativo = imagem.relative_to(DATASET)
        anotacao = imagem.with_suffix(".txt")

        destino_imagem = DATASET_SAIDA / "images" / nome / caminho_relativo
        destino_anotacao = DATASET_SAIDA / "labels" / nome / caminho_relativo.with_suffix(".txt")

        destino_imagem.parent.mkdir(parents=True, exist_ok=True)
        destino_anotacao.parent.mkdir(parents=True, exist_ok=True)

        shutil.copy2(imagem, destino_imagem)
        shutil.copy2(anotacao, destino_anotacao)

for nome, registros_particao in particoes.items():
    print(f"{nome}: {len(registros_particao)} imagens")

Imagens para dividir: 2099
Registros para dividir: 2099
train: 1469 imagens
val: 315 imagens
test: 315 imagens


In [ ]:
from collections import Counter

# Confirma cobertura total e distribuição de classes em cada partição.
total_dividido = sum(len(registros) for registros in particoes.values())
print("Total dividido:", total_dividido)
print("Data original:", len(images))
print("Cobertura completa:", total_dividido == len(images))

for nome, registros_particao in particoes.items():
    classes = Counter()
    for imagem, _ in registros_particao:
        anotacao = imagem.with_suffix(".txt")
        classes.update(
            linha.split()[0]
            for linha in anotacao.read_text().splitlines()
            if linha.strip()
        )
    print(nome, dict(classes))

Total dividido: 2099
Data original: 2099
Cobertura completa: True
train {'0': 2240, '1': 312}
val {'1': 66, '0': 459}
test {'1': 64, '0': 467}


In [ ]:
import csv

manifesto = DATASET_SAIDA / "manifest.csv"

with manifesto.open("w", newline="", encoding="utf-8") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["imagem", "split", "anotacao"])

    for nome, registros_particao in particoes.items():
        for imagem, _ in registros_particao:
            anotacao = imagem.with_suffix(".txt")
            escritor.writerow([
                str(imagem.relative_to(DATASET)),
                nome,
                str(anotacao.relative_to(DATASET)),
            ])

print("Manifesto salvo em:", manifesto)

Manifesto salvo em: c:\Workspace\Study\UPE\urbansense\data\Potholes_dataset_dividido\manifest.csv


In [ ]:
import yaml

configuracao = {
    "path": str(DATASET_SAIDA.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "pothole",
        1: "manhole_cover",
    },
}

yaml_path = DATASET_SAIDA / "data.yaml"
yaml_path.write_text(
    yaml.safe_dump(configuracao, sort_keys=False),
    encoding="utf-8",
)

print(yaml_path.read_text())

path: C:\Workspace\Study\UPE\urbansense\data\Potholes_dataset_dividido
train: images/train
val: images/val
test: images/test
names:
  0: pothole
  1: manhole_cover



In [ ]:
for nome in ("train", "val", "test"):
    imagens_split = sorted(
        (DATASET_SAIDA / "images" / nome).rglob("*.jpg")
    )
    anotacoes_split = sorted(
        (DATASET_SAIDA / "labels" / nome).rglob("*.txt")
    )
    print(nome, len(imagens_split), "imagens,", len(anotacoes_split), "anotações")

train 1469 imagens, 1469 anotações
val 315 imagens, 315 anotações
test 315 imagens, 315 anotações


# Baseline de treino

%pip install ultralytics

In [ ]:
%pip install "ultralytics-platform>=0.1.102" "ultralytics-thop>=2.2.4"

Defaulting to user installation because normal site-packages is not writeable
  Using cached ultralytics_platform-0.1.120-py3-none-any.whl.metadata (18 kB)
  Using cached ultralytics_thop-2.2.5-py3-none-any.whl.metadata (14 kB)
Using cached ultralytics_platform-0.1.120-py3-none-any.whl (104 kB)
Using cached ultralytics_thop-2.2.5-py3-none-any.whl (32 kB)

   -------------------- ------------------- 1/2 [ultralytics-platform]
   -------------------- ------------------- 1/2 [ultralytics-platform]
   ---------------------------------------- 2/2 [ultralytics-platform]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
